In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

sns.set_theme(style='whitegrid')
%matplotlib inline

RAW = "../data/raw/"
PROCESSED = "../data/processed/"



In [2]:
# Load data
user_level_dataset = pd.read_csv(
    PROCESSED + 'user_level_dataset.csv'
)

print(user_level_dataset.shape)

(25000, 36)


## Treatment vs Control Analysis

Objective: Determine whether users exposed to ads converted at a higher rate compared to users who were not.

In [3]:
# How many users are in each group?

pd.crosstab(
    user_level_dataset['exposed_flag'],
    user_level_dataset['converted_flag'],
    margins=True
)

converted_flag,0.00,1.00,All
exposed_flag,,,
0.00,13074,2208,15282
1.00,8276,1442,9718
All,21350,3650,25000


Advertising appears to increase conversion, but is it stastistically significant?

In [4]:
treatment_cr = 1442 / 9718
control_cr = 2208 / 15282

print(f"Treatment CR: {treatment_cr:.2%}")
print(f"Control CR: {control_cr:.2%}")

Treatment CR: 14.84%
Control CR: 14.45%


### Initial Conversion Rate Comparison

Users exposed to advertising converted at a rate of 14.84%, compared to 14.45% for users who were not exposed.

The observed difference is +0.39 percentage points.

While the exposed group exhibits a slightly higher conversion rate, the magnitude of the difference is relatively small and does not by itself establish causal impact.

Further statistical testing is required to determine whether the observed difference is likely attributable to advertising or random variation.

In [5]:
# Calculate Absolute Lift
absolute_lift = (treatment_cr - control_cr) * 100
print('Absolute Lift: ',round(absolute_lift, 2),'%') 

# Calculate Relative Lift
relative_lift = ((treatment_cr / control_cr) - 1 ) * 100
print('Relative Lift: ',round(relative_lift, 2),'%')

Absolute Lift:  0.39 %
Relative Lift:  2.7 %


### Interpretation
Absolute Lift: Users exposed to ads converted .39 percentage points more often than non-exposed users.

Relative Lift: The exposed group converted about 2.7% better than the control group.

In [6]:
# Calculate Incremental Conversions
incremental_conversions = (
    treatment_cr - control_cr
) * 9718

print(incremental_conversions)

37.90734197094642


Incremental Lift seems really small.

Possibility 1 (and biggest factor): 
The synthetic data was generated mostly independently resulting in exposure and conversion being weakly connected. 

Possibility 2: 
Exposure isn't random, maybe exposed users differ from control users.

Possibility 3:
The observed difference could just be noise.

==========================

Is the observed lift stasitically significant?

Null Hypothesis:
Advertising has no effect.
treatment_cr = control_cr

Alternative Hypothesis:
Advertising increased conversion.
treatment_cr > control_cr



In [8]:
# Is the observed difference larger than we'd expect from random variation alone?

# treatment group
treatment_users = (
    user_level_dataset['exposed_flag'] == 1
).sum()

treatment_conversions = (
    (user_level_dataset['exposed_flag'] == 1)
    &
    (user_level_dataset['converted_flag'] == 1)
).sum()

# Control group
control_users = (
    user_level_dataset['exposed_flag'] == 0
).sum()

control_conversions = (
    (user_level_dataset['exposed_flag'] == 0)
    &
    (user_level_dataset['converted_flag'] == 1)
).sum()

print('Treatment Users:', treatment_users)
print('Treatment Conversions:', treatment_conversions)

print('Control Users:', control_users)
print('Control Conversions:', control_conversions)




Treatment Users: 9718
Treatment Conversions: 1442
Control Users: 15282
Control Conversions: 2208


In [9]:
# Summary Table
conversion_summary = (
    user_level_dataset
    .groupby("exposed_flag")
    .agg(
        users=("user_id", "count"),
        conversions=("converted_flag", "sum")
    )
)

conversion_summary["conversion_rate"] = (
    conversion_summary["conversions"]
    / conversion_summary["users"]
)

conversion_summary

,users,conversions,conversion_rate
exposed_flag,,,
0.00,15282,"2,208.00",0.14
1.00,9718,"1,442.00",0.15


In [11]:
# Z-test
p1 = treatment_conversions / treatment_users
p2 = control_conversions / control_users

p_pool = (
    treatment_conversions + control_conversions
) / (
    treatment_users + control_users
)

se = np.sqrt(
    p_pool * (1 - p_pool)
    * (
        (1 / treatment_users)
        + (1 / control_users)
    )
)

z_stat = (p1 - p2) / se

print('Z-statistic:', z_stat)

Z-statistic: 0.8514313085924398


In [12]:
# Calculate P-Value
from scipy.stats import norm

p_value = 1 - norm.cdf(z_stat)

print('P-value:', p_value)

P-value: 0.19726490303837307


### Statistial Significance Test
Because the p-value exceeds the 0.05 significance threshold, we fail to reject the null hypothesis.

While exposed users converted at a slightly higher rate than non-exposed users, the observed difference is not statistically significant and may be attributable to random variation.

====================

### Take Away
The campaign showed a directional lift of 2.7%, but the result was not statistically significant (p=0.197), so we cannot confidently attribute the observed difference to advertising.

### Treatment vs Control Balance Check

Objective: 
Determine whether exposed and non-exposed users are comparable before measuring incremental lift.

Question: 
Are exposed users different from non-exposed users?

In [ ]:
# Were exposed users more likely to be prior subscribers?

pd.crosstab(
    user_level_dataset['exposed_flag'],
    user_level_dataset['prior_subscriber_flag'],
    normalize='index'
).round(4)

prior_subscriber_flag,0,1
exposed_flag,,
0.00,0.70,0.30
1.00,0.60,0.40


In [14]:
# Were exposed users more likely to have trialed before?

pd.crosstab(
    user_level_dataset['exposed_flag'],
    user_level_dataset['prior_trial_flag'],
    normalize='index'
).round(4)


prior_trial_flag,0,1
exposed_flag,,
0.00,0.69,0.31
1.00,0.63,0.37


In [15]:
# Did the campaign target different customer types?

pd.crosstab(
    user_level_dataset['exposed_flag'],
    user_level_dataset['customer_segment'],
    normalize='index'
).round(4)

customer_segment,Casual Streamer,Heavy Streamer,Lapsed Subscriber,New Prospect,Premium Loyalist,Trialist,Win-back Target
exposed_flag,,,,,,,
0.00,0.31,0.07,0.10,0.19,0.14,0.12,0.05
1.00,0.26,0.07,0.14,0.15,0.17,0.11,0.09


In [16]:
# Were exposed users more engaged before the campaign?

user_level_dataset.groupby(
    'exposed_flag'
)['historical_engagement_score'].agg(
    ['count','mean','median','std']
)

,count,mean,median,std
exposed_flag,,,,
0.00,15282,35.46,33.50,16.85
1.00,9718,37.87,36.10,17.52


### Treatment-Control Balance Assessment

Several meaningful imbalances were observed. Exposed users were more likely to be prior subscribers (40% vs 30%) and prior trialists (37% vs 31%) than non-exposed users. The treatment group also contained higher concentrations of Lapsed Subscribers, Premium Loyalists, and Win-back Targets, while the control group contained a larger share of New Prospects.

Historical engagement was also slightly higher among exposed users (37.9 vs 35.5 average engagement score).

These findings suggest that exposed users may have been more predisposed to convert before advertising exposure occured. As a result, the simple treatment-versus-control lift estimate may overstate the true casual impact of advertising dur to selection bias.

Additional causal inference methods such as propensity score matching, regression adjustment, or randomized holdout testing would improve confidence in the estimated incremental effect.